# The machine learning workflow

Every model in this section follows the same loop: split the data, fit on one part, measure on another, adjust, repeat. Get this loop wrong and every later number is flattering you. This notebook covers the loop on its own, with a plain linear model and California housing, so the later notebooks can focus on the models.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

from common.datasets import load_california_housing

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

A model is a function with knobs (parameters). Training turns the knobs until the function matches the examples we gave it. That part is easy, and a big enough model can match any set of examples perfectly by memorising them.

What we actually want is good predictions on data the model has not seen. So the real question is never "how well does it fit the training data?" It is "how well will it do tomorrow?" Everything in this notebook is a way of estimating that honestly.

An analogy: a student who memorises last year's exam gets full marks on last year's exam. Only a fresh exam tells you what they understand.

## 2. Maths

**Loss and cost.** A loss measures the error on one example. The cost is the average loss over the data. For regression the usual choice is squared error:

$$J(w)=\frac{1}{n}\sum_{i=1}^{n}\bigl(y_i-\hat y_i\bigr)^2,\qquad \hat y_i=f(x_i;w)$$

Training means finding $w$ that makes $J$ small. This is why the choice of loss matters. It is the only thing the model is trying to do. Section 02 showed that squared error is what Gaussian noise gives you.

**Generalisation error.** What we care about is the expected loss on new data from the same source:

$$\text{generalisation error}=\mathbb{E}_{(x,y)}\bigl[(y-f(x;\hat w))^2\bigr]$$

We cannot compute this, so we estimate it on held-out data the model never trained on.

**Three sets, three jobs.**

- Train: fit the parameters.
- Validation: compare models and tune settings (hyperparameters).
- Test: one final, honest estimate. Touch it once.

**Cross-validation** (k-fold) repeats the train/validation split $k$ times so every row is used for validation once. The average of the $k$ scores is a steadier estimate than a single split, and the spread tells you how much to trust it.

## 3. The data

California housing: 20,640 census blocks, 8 numeric features, target is the median house value in units of $100,000. I take a 4,000-row sample so every notebook stays quick.

In [ ]:
X, y = load_california_housing(n_samples=4000, seed=SEED)
print(X.shape)
X.head()

In [ ]:
print(y.describe())
X.describe().loc[["mean", "std", "min", "max"]]

Note the scales: `Population` runs to thousands while `MedInc` is a few units. That matters for gradient descent in notebook 10.

## 4. Split the data

Split once into train (60%), validation (20%) and test (20%). Fix the seed so the split is the same every run. Put the test set aside and do not look at it until the end.

In [ ]:
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=SEED)
print(len(X_train), len(X_val), len(X_test))

## 5. The cost function in action

First, what does "good" mean? Always compare against a baseline. The simplest model predicts the training mean for every row. Any real model has to beat it.

The from-scratch cost is two lines.

In [ ]:
def mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))


baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
print("baseline val MSE:", mse(y_val, baseline.predict(X_val)))
print("sklearn agrees  :", mean_squared_error(y_val, baseline.predict(X_val)))

Now a one-parameter model, $\hat y = w \cdot \text{MedInc}$ (no intercept, to keep it one knob). Sweep $w$ and plot the cost. Training is the search for the bottom of this curve. For squared error it is a bowl with one minimum, which is why linear regression is easy to optimise.

In [ ]:
inc = X_train["MedInc"].to_numpy()
ws = np.linspace(0, 1.2, 100)
costs = [mse(y_train, w * inc) for w in ws]
best = ws[int(np.argmin(costs))]

plt.plot(ws, costs)
plt.axvline(best, ls="--", c="grey")
plt.xlabel("w")
plt.ylabel("training cost (MSE)")
plt.title(f"Cost bowl, minimum near w = {best:.2f}")
plt.show()

## 6. Fit, evaluate, tune

A plain linear model on all 8 features beats the baseline by a wide margin.

In [ ]:
lin = LinearRegression().fit(X_train, y_train)
print("train MSE:", mse(y_train, lin.predict(X_train)))
print("val MSE  :", mse(y_val, lin.predict(X_val)))

Train and validation errors are close, so this model is not overfitting. It is probably too simple (underfitting), since neither error is very low.

To see overfitting, use a model with a dial for flexibility. A decision tree (treated as a black box here, Part B covers it) gets more flexible as `max_depth` grows. Fit trees of increasing depth and record both errors.

In [ ]:
depths = range(1, 21)
train_err, val_err = [], []
for d in depths:
    tree = DecisionTreeRegressor(max_depth=d, random_state=SEED).fit(X_train, y_train)
    train_err.append(mse(y_train, tree.predict(X_train)))
    val_err.append(mse(y_val, tree.predict(X_val)))

plt.plot(depths, train_err, label="train")
plt.plot(depths, val_err, label="validation")
plt.xlabel("max_depth")
plt.ylabel("MSE")
plt.legend()
plt.show()
best_depth = list(depths)[int(np.argmin(val_err))]
print("best depth on validation:", best_depth, " val MSE:", min(val_err))
print("depth 20 train MSE:", train_err[-1], " val MSE:", val_err[-1])

This is the shape to remember. Training error only ever falls as the model gets more flexible. Validation error falls, bottoms out, then rises. The gap between the two lines is overfitting: the model has started to memorise noise.

Left of the minimum the model underfits (too simple). Right of it, it overfits (too flexible). Tuning is the search for the minimum, and the validation set is what lets us see it.

## 7. Cross-validation

One validation split is a single roll of the dice. A different split could pick a different depth. K-fold cross-validation averages over several splits. Here is what the splitting does, using 5 folds on 10 rows.

In [ ]:
for k, (tr, va) in enumerate(KFold(5, shuffle=True, random_state=SEED).split(np.arange(10))):
    print(f"fold {k}: train {tr}  validate {va}")

Now score the linear model and a few tree depths using 5-fold CV on the train+validation data (the test set stays locked away). `cross_val_score` returns negative MSE because sklearn treats higher as better.

In [ ]:
cv = KFold(5, shuffle=True, random_state=SEED)
scores = -cross_val_score(LinearRegression(), X_tmp, y_tmp, cv=cv, scoring="neg_mean_squared_error")
print("linear folds:", scores, " mean:", scores.mean(), " sd:", scores.std())

cv_by_depth = {}
for d in (3, 5, 7, 10, 15):
    tree = DecisionTreeRegressor(max_depth=d, random_state=SEED)
    s = -cross_val_score(tree, X_tmp, y_tmp, cv=cv, scoring="neg_mean_squared_error")
    cv_by_depth[d] = s.mean()
    print(f"depth {d:>2}: CV MSE {s.mean():.4f} +/- {s.std():.4f}")

The fold-to-fold spread is real. Differences smaller than it are noise, so do not read much into a 0.01 gap between two settings.

The loop is now complete: choose the setting with the best CV score, refit on all train+validation data, and score once on the test set.

In [ ]:
best_d = min(cv_by_depth, key=cv_by_depth.get)
final = DecisionTreeRegressor(max_depth=best_d, random_state=SEED).fit(X_tmp, y_tmp)
print("chosen depth:", best_d)
print("test MSE    :", mse(y_test, final.predict(X_test)))

## 8. What breaks: data leakage

Leakage means information from outside the training data sneaks into the fit, so your estimate is too optimistic and the model disappoints in production. The usual cause is doing a preprocessing step on the whole dataset before splitting.

Here is an extreme case. The target is pure noise, unrelated to any feature. No model can do better than guessing, so a fair score is about $R^2 = 0$. But if we pick the "best" 10 of 1,000 random features using all the labels first, and cross-validate afterwards, the selection step has already seen the answers.

In [ ]:
n, p = 100, 1000
X_noise = rng.normal(size=(n, p))
y_noise = rng.normal(size=n)
cv = KFold(5, shuffle=True, random_state=SEED)

# Wrong: select on everything, then cross-validate
X_sel = SelectKBest(f_regression, k=10).fit_transform(X_noise, y_noise)
leaky = cross_val_score(LinearRegression(), X_sel, y_noise, cv=cv, scoring="r2").mean()

# Right: selection is a step inside the pipeline, so it refits on each training fold
pipe = make_pipeline(SelectKBest(f_regression, k=10), LinearRegression())
honest = cross_val_score(pipe, X_noise, y_noise, cv=cv, scoring="r2").mean()

print("leaky R^2 :", leaky)
print("honest R^2:", honest)

The leaky score looks like a real model. The honest score is at or below zero, which is the truth. The same applies to something as ordinary as a scaler: fitting `StandardScaler` on all rows lets the test set's mean and spread leak into training. The effect is small there but the habit is the same.

A `Pipeline` fixes this by construction. Inside `cross_val_score`, each fold calls `fit` on the training part only, then applies the already-fitted steps to the validation part.

In [ ]:
model = make_pipeline(StandardScaler(), LinearRegression())
s = -cross_val_score(model, X_tmp, y_tmp, cv=cv, scoring="neg_mean_squared_error")
print("scaled linear CV MSE:", s.mean())

Rules I use:

- Split first. Every step that learns something from data (scaling, imputing, feature selection, encoding) goes inside the pipeline.
- Touch the test set once.
- Group related rows (the same customer, the same house) into the same fold, or the model memorises them.
- If a score looks too good, suspect leakage before celebrating.

## Summary

- The cost function is the model's only goal. Choose it deliberately.
- Training error always improves with flexibility. Validation error is the one to watch.
- Cross-validation gives a steadier estimate and a spread. Test once at the end.
- Pipelines stop leakage. A suspiciously good score is more often a leak than a breakthrough.

Next: `10_linear_regression.ipynb` builds the first model properly.